# Stage 17b — lexicon-constrained decode on Stage 11 (the sequence model that works)
The trajectory-IMAGE approach was a NO-GO; the trajectory *sequence* reads at
**0.43** (Stage 11 Conformer+CTC). Here we add a **closed-vocabulary language
prior** to its CTC output for the **lex** subset: score every TRAIN-lex word
under the model (exact CTC forward) and pick the most probable valid word.
Soft-fallback (greedy is always a candidate) so it **can only help, never hurt**.
nonlex stays greedy. CPU is fine.

**Attach:** `gaurs86/wita-full-english-landmark-cache` + your **Stage 11 checkpoint**
dataset. Internet ON (for editdistance).


## Cell 1 — clone + deps


In [ ]:
import sys, subprocess as sp
sp.run('rm -rf /kaggle/working/wita_v2', shell=True)
sp.run("git clone -b stage13b-paper-replication "
       "'https://github.com/Gaurs86/WiTA-v2.git' '/kaggle/working/wita_v2'",
       shell=True, check=True)
sys.path.insert(0, '/kaggle/working/wita_v2')
for _m in [m for m in sys.modules if m.split('.')[0] in ('stage16','stage17','datasets','models')]:
    del sys.modules[_m]
sp.run('pip -q install editdistance', shell=True)
import torch; print('torch', torch.__version__, '| GPU', torch.cuda.is_available())


## Cell 2 — locate landmark cache + Stage 11 checkpoint
If the checkpoint probe is wrong, set CKPT manually.


In [ ]:
import os, glob
from stage17.common import find_landmark_cache
LM_CACHE = find_landmark_cache(preferred=[
    '/kaggle/input/datasets/gaurs86/wita-full-english-landmark-cache'])
print('landmark cache:', LM_CACHE, '| ok:', os.path.isdir(os.path.join(LM_CACHE,'train')))

# probe for the Stage 11 checkpoint (.pth/.pt under /kaggle/input)
cands = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True)
             + glob.glob('/kaggle/input/**/*.pt',  recursive=True))
print('checkpoint candidates:'); [print('  ', c) for c in cands[:20]]
pref = [c for c in cands if 'stage11' in c.lower() or 'best' in c.lower()]
CKPT = (pref or cands)[0] if (pref or cands) else ''
CKPT = CKPT  # <-- set manually if the auto-pick is wrong
print('USING CKPT:', CKPT)


## Cell 3 — load model + build lexicon (TRAIN lex words only)


In [ ]:
import torch
from stage17.seq_lexicon_decode import CTCConverter, load_stage11, build_lexicon_from_cache
device = 'cuda' if torch.cuda.is_available() else 'cpu'
conv = CTCConverter()
model = load_stage11(CKPT, device)          # prints best_payload (sanity-check the CER) + key match
by_fc, lex_words = build_lexicon_from_cache(LM_CACHE, conv, wordfreq_topk=0)


## Cell 4 — VAL: greedy vs lexicon (free diagnostic; tune len_window)
Watch **coverage** — if ~100%, the TRAIN lexicon covers val lex words (closed
vocab applies -> big potential gain). Also confirm **greedy lex ~0.43**: if it's
~1.0 the model arch didn't match the checkpoint (fix d_model/n_layers).


In [ ]:
from stage17.seq_lexicon_decode import evaluate_split, print_result
best = None
for lw in (2, 4, 8):
    r = evaluate_split(model, LM_CACHE, 'val', conv, by_fc, lex_words, device, len_window=lw)
    print_result(r)
    if best is None or r['lexicon']['overall'] < best['lexicon']['overall']:
        best = r
BEST_LW = best['len_window']
print('BEST len_window on val =', BEST_LW, '-> val lexicon overall', round(best['lexicon']['overall'],4))


## Cell 5 — TEST: run ONCE with the val-best len_window (marker-gated)


In [ ]:
import json
MARKER = '/kaggle/working/.stage17_lexicon_test_evaluated'
assert not os.path.exists(MARKER), 'Test already evaluated once. Delete marker only if intentional.'
res = evaluate_split(model, LM_CACHE, 'test', conv, by_fc, lex_words, device, len_window=BEST_LW)
res['val_best_len_window'] = BEST_LW
json.dump(res, open('/kaggle/working/stage17_lexicon_test.json','w'), indent=2, default=float)
open(MARKER,'w').write('done')
print_result(res)


## Results
| Decoder | lex | nonlex | overall |
|---|---|---|---|
| Stage 11 greedy | ~0.435 | ~0.545 | ~0.450 |
| Stage 11 + lexicon (lex) | … | (=greedy) | … |
| Paper baseline | 0.281 | 0.365 | 0.2924 |

Lexicon decoding only touches **lex**; nonlex is unchanged (greedy). The lex
improvement is the headline. If val coverage was ~100% and the gain is large,
this is a closed-vocabulary result (report it as such, honestly) — train-derived
lexicon, no test-label leakage.
